In [1]:
import numpy as np
import jax
import jax.numpy as jnp
from jax import jacfwd
from jax import jit
from jax import vmap
from functools import partial
import sys
sys.path.append("../src/gapmoe/")
import calc_vEarth
import EarthMotion

In [2]:
tref = 10063.874
RA_str, Dec_str = "17:57:38.03","-28:38:28.53"
RA_deg = calc_vEarth.hms_string_to_degrees(RA_str)
Dec_deg = calc_vEarth.dms_string_to_degrees(Dec_str)
vEarth = EarthMotion.calc_vEarth(2450000+tref,RA_deg,Dec_deg)

In [3]:
t0 = 10090 - 5       
u0 = 0.01          
q = 0.001
alpha = np.pi/2 + 1
tE = 47.1147
rho = 0.000528216
s = 1.22626
piEN = 0.0565383
piEE = 0.0565382
gamma1 = 0.000702119
gamma2 = 0.00175531
gamma3 = 0.0017553
thS = 0.0001800539

In [4]:
@jit
def physical_to_lightcurve_circular_old(theta,thS):
    tE     = theta[1]
    rho    = theta[3]
    s      = theta[5]
    piEN   = theta[7]
    piEE   = theta[8]
    gamma1 = theta[9]
    gamma2 = theta[10]
    gamma3 = theta[11]

    piE = jnp.sqrt(piEN**2 + piEE**2)

    thE = thS / rho
    ML = thE / 8.1439 / piE # KAPPA = 8.1429 [mas / Msun]
    murel = thE / tE * 365.25
    murel_E = murel * piEE / piE
    murel_N = murel * piEN / piE

    gamma_sq = gamma1**2 + gamma2**2 + gamma3**2
    gamma_parallel = jnp.sqrt(gamma1**2 + gamma3**2)
    gamma_ratio = jnp.sqrt(1 + (gamma1/gamma3)**2)
    orbital_scale = jnp.cbrt((s**3) * gamma_sq * gamma_ratio / (ML * 2.959122082855911e-4)) # G = 2.959122082855911e-4 [AU^3 / (Msun * day^2)]
    gamma_abs = jnp.sqrt(gamma_sq)
    Ds = 1 / ((orbital_scale - piE) * thE)

    pi_rel = thE * piE
    pi_S = 1 / Ds
    pi_L = pi_rel + pi_S
    DL = 1 / pi_L

    RE = DL * thE
    orbital_radi = RE * s * gamma_ratio

    cosi = gamma2 / (gamma_ratio * gamma_abs)
    tanphi = - gamma1 * gamma_abs / (gamma3 * gamma_parallel)

    return  jnp.array([murel_N, murel_E, ML, DL, Ds, orbital_radi, cosi, tanphi])

@jit
def wrapped_physical_to_lightcurve_circular_old(theta_reduced, thS):
    tE, rho, s, piEN, piEE, gamma1, gamma2, gamma3 = theta_reduced

    piE = jnp.sqrt(piEE**2 + piEN**2)
    thE = thS / rho
    ML = thE / 8.1439 / piE
    murel = thE / tE * 365.25
    murel_E = murel * piEE / piE
    murel_N = murel * piEN / piE

    gamma_sq = gamma1**2 + gamma2**2 + gamma3**2
    gamma_parallel = jnp.sqrt(gamma1**2 + gamma3**2)
    gamma_ratio = jnp.sqrt(1 + (gamma1/gamma3)**2)
    orbital_scale = jnp.cbrt((s**3) * gamma_sq * gamma_ratio / (ML * 2.959122082855911e-4))
    gamma_abs = jnp.sqrt(gamma_sq)
    Ds = 1 / ((orbital_scale - piE) * thE)

    pi_rel = thE * piE
    pi_S = 1 / Ds
    pi_L = pi_rel + pi_S
    DL = 1 / pi_L

    RE = DL * thE
    orbital_radi = RE * s * gamma_ratio

    cosi = gamma2 / (gamma_ratio * gamma_abs)
    tanphi = - gamma1 * gamma_abs / (gamma3 * gamma_parallel)

    return jnp.array([murel_N, murel_E, ML, DL, Ds, orbital_radi, cosi, tanphi])

@jit
def calc_ln_det_jacobian_circular_old(theta, thS):
    theta_reduced = jnp.array([
        theta[1],
        theta[3],
        theta[5],
        theta[7],
        theta[8],
        theta[9],
        theta[10],
        theta[11]
    ])
    J = jacfwd(wrapped_physical_to_lightcurve_circular_old)(theta_reduced, thS)
    sign, lndet = jnp.linalg.slogdet(J)
    return lndet

In [5]:
x =  jnp.array([t0, tE, u0, rho, q, s, alpha, piEN, piEE, gamma1, gamma2, gamma3])
murel_N, murel_E, ML, DL, Ds, orbital_radi, cosi, tanphi = physical_to_lightcurve_circular_old(x, thS)

print(f"μ_rel_E   = {murel_E:.6f}  # mas/yr")
print(f"μ_rel_N   = {murel_N:.6f}  # mas/yr")
print(f"M_L       = {ML:.6f}       # M_sun")
print(f"D_L       = {DL:.6f}       # kpc")
print(f"D_S       = {Ds:.6f}       # kpc")
print(f"R_orbit   = {orbital_radi:.6f}  # AU")
print(f"cos(i)    = {cosi:.6f}")
print(f"tan(φ)    = {tanphi:.6f}")

print(calc_ln_det_jacobian_circular_old(x, thS))

μ_rel_E   = 1.868570  # mas/yr
μ_rel_N   = 1.868573  # mas/yr
M_L       = 0.523481       # M_sun
D_L       = 6.663750       # kpc
D_S       = 8.142625       # kpc
R_orbit   = 2.999999  # AU
cos(i)    = 0.631751
tan(φ)    = -0.545831
32.090496


In [6]:
@jit
def physical_to_lightcurve_circular(theta,thS,vEarth):
    t0 = theta[0]
    tE     = theta[1]
    u0 = theta[2]
    rho    = theta[3]
    q = theta[4]
    s      = theta[5]
    alpha = theta[6]
    piEN   = theta[7]
    piEE   = theta[8]
    gamma1 = theta[9]
    gamma2 = theta[10]
    gamma3 = theta[11]
    vEarth_N, vEarthE = vEarth
    
    G = 2.959122082855911e-4 # [AU^3 / (Msun * day^2)]
    KAPPA = 8.1429 # [mas / Msun]

    piE = jnp.sqrt(piEN**2 + piEE**2)

    thE = thS / rho #mas
    ML = thE / KAPPA / piE #Msun
    murel_geo = thE / tE * 365.25 # mas / year
    murel_N_geo = murel_geo * piEN / piE # mas / year
    murel_E_geo = murel_geo * piEE / piE # mas / year

    gamma_sq = gamma1**2 + gamma2**2 + gamma3**2
    gamma_parallel = jnp.sqrt(gamma1**2 + gamma3**2)
    gamma_ratio = jnp.sqrt(1 + (gamma1/gamma3)**2)
    orbital_scale = jnp.cbrt((s**3) * gamma_sq * gamma_ratio / (ML * G))
    gamma_abs = jnp.sqrt(gamma_sq)
    Ds = 1 / ((orbital_scale - piE) * thE) #kpc

    pi_rel = thE * piE
    pi_S = 1 / Ds
    pi_L = pi_rel + pi_S
    DL = 1 / pi_L #kpc
    
    murel_N_hel = murel_N_geo + thE * piE * vEarth[0]
    murel_E_hel = murel_E_geo + thE * piE * vEarth[1]
    
    RE = DL * thE # AU
    orbital_radi = RE * s * gamma_ratio #AU

    r = RE * s * jnp.array([1,0, - gamma1 / gamma3]) #AU
    v = RE * s * jnp.array([gamma1,gamma2,gamma3]) #AU / day
    
    h = jnp.cross(r,v)
    
    z = h / jnp.sqrt(jnp.dot(h,h))
    
    cos_i = z[2]
    sin_i = jnp.sqrt(1 - cos_i**2)
    sin_Om0, cos_Om0 = z[0] / sin_i, - z[1] / sin_i
    Om0 = jnp.arctan2(sin_Om0, cos_Om0)
    Om_NE = Om0 + jnp.arctan2(piEE,piEN) + np.pi - alpha
    
    x = jnp.array([cos_Om0,sin_Om0,0])
    y = jnp.cross(z,x)
    
    cos_phi0 = jnp.dot(r,x)/jnp.sqrt(jnp.dot(r,r))
    sin_phi0 = jnp.dot(r,y)/jnp.sqrt(jnp.dot(r,r))
    phi0 = jnp.arctan2(sin_phi0,cos_phi0)

    return  jnp.array([t0, u0, q, ML, DL, Ds, murel_N_hel, murel_E_hel, orbital_radi, cos_i, Om_NE, phi0])

@jit
def calc_ln_det_jacobian_circular(theta, thS,vEarth):
    J = jacfwd(physical_to_lightcurve_circular)(theta, thS,vEarth)
    sign, lndet = jnp.linalg.slogdet(J)
    return lndet

In [7]:
t0, u0, q, ML, DL, Ds, murel_N_hel, murel_E_hel, orbital_radi, cos_i, Om_NE, phi0  = physical_to_lightcurve_circular(x,thS,vEarth)
    
print(f"μ_rel_E   = {murel_E:.6f}  # mas/yr")
print(f"μ_rel_N   = {murel_N:.6f}  # mas/yr")
print(f"M_L       = {ML:.6f}       # M_sun")
print(f"D_L       = {DL:.6f}       # kpc")
print(f"D_S       = {Ds:.6f}       # kpc")
print(f"R_orbit   = {orbital_radi:.6f}  # AU")
print(f"cos(i)    = {cosi:.6f}")
print(f"tan(φ)    = {np.tan(phi0):.6f}")
print(calc_ln_det_jacobian_circular(x, thS,vEarth))

μ_rel_E   = 1.868570  # mas/yr
μ_rel_N   = 1.868573  # mas/yr
M_L       = 0.523546       # M_sun
D_L       = 6.664022       # kpc
D_S       = 8.143031       # kpc
R_orbit   = 3.000122  # AU
cos(i)    = 0.631751
tan(φ)    = -0.545831
31.83003


In [8]:
@jit
def physical_to_lightcurve_kepler(theta,thS,vEarth):
    t0 = theta[0]
    tE     = theta[1]
    u0 = theta[2]
    rho    = theta[3]
    q = theta[4]
    s      = theta[5]
    alpha = theta[6]
    piEN   = theta[7]
    piEE   = theta[8]
    gamma1 = theta[9]
    gamma2 = theta[10]
    gamma3 = theta[11]
    r_s = theta[12]
    a_s = theta[13] 
    vEarth_N, vEarthE = vEarth
    
    G = 2.959122082855911e-4 # [AU^3 / (Msun * day^2)]
    KAPPA = 8.1429 # [mas / Msun]

    piE = jnp.sqrt(piEN**2 + piEE**2)

    thE = thS / rho #mas
    ML = thE / KAPPA / piE #Msun
    murel_geo = thE / tE * 365.25 # mas / year
    murel_N_geo = murel_geo * piEN / piE # mas / year
    murel_E_geo = murel_geo * piEE / piE # mas / year


    gamma_sq = gamma1**2 + gamma2**2 + gamma3**2
    gamma_abs = jnp.sqrt(gamma_sq)
    a_norm = a_s * s * jnp.sqrt(1 + r_s**2)
    orbital_scale = jnp.cbrt((s**2) * a_norm * gamma_sq / (ML * G) / (2 * a_s - 1))
    orbital_scale = jnp.cbrt((s**3) * a_s * jnp.sqrt(1 + r_s**2) * gamma_sq / (ML * G) / (2 * a_s - 1))
    Ds = 1 / ((orbital_scale - piE) * thE) #kpc

    pi_rel = thE * piE
    pi_S = 1 / Ds
    pi_L = pi_rel + pi_S
    DL = 1 / pi_L #kpc
    
    murel_N_hel = murel_N_geo + thE * piE * vEarth[0]
    murel_E_hel = murel_E_geo + thE * piE * vEarth[1]

    RE = DL * thE # AU
    orbital_radi = RE * a_norm #AU

    r = RE * s * jnp.array([1,0, r_s]) #AU
    v = RE * s * jnp.array([gamma1,gamma2,gamma3]) #AU / day
    
    h = jnp.cross(r,v)
    A = jnp.cross(v,h) / (G*ML) - r / jnp.sqrt(jnp.dot(r,r))
    e = jnp.sqrt(jnp.dot(A,A))
    
    z = h / jnp.sqrt(jnp.dot(h,h))
    x = A / e
    y = jnp.cross(z,x)
    
    cos_i = z[2]
    sin_i = jnp.sqrt(1 - cos_i**2)
    
    sin_Om0, cos_Om0 = z[0] / sin_i, - z[1] / sin_i
    Om0 = jnp.arctan2(sin_Om0, cos_Om0)
    Om_NE = Om0 + jnp.arctan2(piEE,piEN) + jnp.pi - alpha
    
    sin_om, cos_om = x[2] / sin_i, y[2] / sin_i
    om = jnp.arctan2(sin_om, cos_om)
    
    cos_nu = jnp.dot(r,x)/jnp.sqrt(jnp.dot(r,r))
    sin_nu = jnp.dot(r,y)/jnp.sqrt(jnp.dot(r,r))
    nu = jnp.arctan2(sin_nu,cos_nu)

    return  jnp.array([t0, u0, q, ML, DL, Ds, murel_N_hel, murel_E_hel, orbital_radi, e, cos_i, Om_NE, om, nu])

@jit
def calc_ln_det_jacobian_kepler(theta, thS,vEarth):
    J = jacfwd(physical_to_lightcurve_kepler)(theta, thS,vEarth)
    sign, lndet = jnp.linalg.slogdet(J)
    return lndet

In [9]:
r_s = 1
a_s = 1.3
x_kep =  jnp.array([t0, tE, u0, rho, q, s, alpha, piEN, piEE, gamma1, gamma2, gamma3,r_s,a_s])
t0, u0, q, ML, DL, Ds, murel_N_hel, murel_E_hel, orbital_radi, e, cos_i, Om_NE, om, nu = physical_to_lightcurve_kepler(x_kep,thS,vEarth)

print(f"μ_rel_E   = {murel_E:.6f}  # mas/yr")
print(f"μ_rel_N   = {murel_N:.6f}  # mas/yr")
print(f"M_L       = {ML:.6f}       # M_sun")
print(f"D_L       = {DL:.6f}       # kpc")
print(f"D_S       = {Ds:.6f}       # kpc")
print(f"R_orbit   = {orbital_radi:.6f}  # AU")
print(f"e    = {e:.6f}")
print(f"cos(i)    = {cosi:.6f}")
print(f"0mega    = {Om_NE:.6f}")
print(f"omega    = {om:.6f}")
print(f"nu    = {nu:.6f}")
print(calc_ln_det_jacobian_kepler(x_kep, thS,vEarth))


μ_rel_E   = 1.868570  # mas/yr
μ_rel_N   = 1.868573  # mas/yr
M_L       = 0.523546       # M_sun
D_L       = 6.521787       # kpc
D_S       = 7.931656       # kpc
R_orbit   = 5.011855  # AU
e    = 0.694834
cos(i)    = 0.631751
0mega    = 0.325815
omega    = -0.863367
nu    = 2.061816
30.737347
